# SynthRare — Healthcare CTGAN Training (Google Colab)

Trains a CTGAN on FHIR-compatible synthetic patient records and uploads to HuggingFace Hub.

In [ ]:
# Cell 1 — Install dependencies + mount Drive
!pip install sdv ctgan huggingface_hub pandas scikit-learn scipy
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Load seed data from Drive
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/synthrare/seed_data/healthcare_seed.csv')
print(f'Loaded {len(df)} rows, {len(df.columns)} columns')
df.dtypes

In [ ]:
# Cell 3 — Train CTGAN model
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)
if 'patient_id' in df.columns:
    metadata.update_column('patient_id', sdtype='id')
    metadata.set_primary_key('patient_id')

synthesizer = CTGANSynthesizer(metadata, epochs=300, verbose=True)
synthesizer.fit(df)
print('Training complete')

In [ ]:
# Cell 4 — Validate output + FHIR domain constraints
from scipy import stats
import numpy as np

synthetic = synthesizer.sample(num_rows=1000)
num_cols = df.select_dtypes('number').columns.tolist()

ks_scores = []
for col in num_cols:
    ks, _ = stats.ks_2samp(df[col].dropna(), synthetic[col].dropna())
    ks_scores.append(ks)
    print(f'  {col}: KS={ks:.4f}')

print(f'Mean KS: {np.mean(ks_scores):.4f}')

# Domain constraint checks
if 'age' in synthetic.columns:
    assert (synthetic['age'].between(0, 130)).all(), 'Age out of range'
if 'systolic_bp' in synthetic.columns:
    assert (synthetic['systolic_bp'] > 0).all(), 'BP must be positive'
if 'bmi' in synthetic.columns:
    assert (synthetic['bmi'].between(10, 80)).all(), 'BMI out of range'
print('Domain constraints passed')

In [ ]:
# Cell 5 — Save and upload to HuggingFace Hub
from huggingface_hub import HfApi

HF_TOKEN = 'YOUR_HF_TOKEN_HERE'
REPO_ID = 'synthrare/healthcare-ctgan'

synthesizer.save('/content/model.pkl')
HfApi().upload_file(
    path_or_fileobj='/content/model.pkl',
    path_in_repo='model.pkl',
    repo_id=REPO_ID,
    token=HF_TOKEN,
)
print(f'Model uploaded to {REPO_ID}')